In [29]:
# IMPORT LIBRARIES

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import math

In [30]:
# CONFIGURATION

ORDERS_PER_YEAR = 250_500
SHIPPED_PER_YEAR = 192_885
START_ORDER_NUM = 2000000

np.random.seed(42)
random.seed(42)

# STORE MASTER WITH CITY & STATE
# FIRST 10 = SHIPPING + BOPIS
# LAST 15 = BOPIS ONLY

STORES = [
    ("Store A","Minneapolis","MN","55401",44.9833,-93.2667),
    ("Store B","Chicago","IL","60601",41.8837,-87.6233),
    ("Store C","Detroit","MI","48226",42.3314,-83.0458),
    ("Store D","Pittsburgh","PA","15222",40.4406,-79.9959),
    ("Store E","Atlanta","GA","30303",33.7490,-84.3880),
    ("Store F","Nashville","TN","37219",36.1627,-86.7816),
    ("Store G","Jacksonville","FL","32202",30.3322,-81.6557),
    ("Store H","Tampa","FL","33602",27.9506,-82.4572),
    ("Store I","Charlotte","NC","28202",35.2271,-80.8431),
    ("Store J","Boston","MA","02108",42.3601,-71.0589),

    # BOPIS ONLY STORES

    ("Store K","Cleveland","OH","44114",41.4993,-81.6944),
    ("Store L","Buffalo","NY","14202",42.8864,-78.8784),
    ("Store M","Richmond","VA","23219",37.5407,-77.4360),
    ("Store N","Raleigh","NC","27601",35.7796,-78.6382),
    ("Store O","Columbia","SC","29201",34.0007,-81.0348),
    ("Store P","Birmingham","AL","35203",33.5186,-86.8104),
    ("Store Q","Louisville","KY","40202",38.2527,-85.7585),
    ("Store R","Indianapolis","IN","46204",39.7684,-86.1581),
    ("Store S","Madison","WI","53703",43.0731,-89.4012),
    ("Store T","Grand Rapids","MI","49503",42.9634,-85.6681),
    ("Store U","Toledo","OH","43604",41.6528,-83.5379),
    ("Store V","Harrisburg","PA","17101",40.2732,-76.8867),
    ("Store W","Albany","NY","12207",42.6526,-73.7562),
    ("Store X","Providence","RI","02903",41.8240,-71.4128),
    ("Store Y","Portland","ME","04101",43.6591,-70.2568),
]

DEPARTMENTS = [f"Dept_{i+1}" for i in range(25)]

In [31]:
# CATALOG GENERATION

def create_catalog(n_skus=43106):

    skus = [f"SKU{i:06}" for i in range(1, n_skus+1)]

    catalog = pd.DataFrame({
        "sku": skus,
        "sku_description": [f"sku_description_{i}" for i in range(1, n_skus+1)],
        "price": np.round(np.random.uniform(3,800,n_skus),2),
        "weight_lb": np.round(np.random.uniform(0.1,200,n_skus),2),
        "length_in": np.round(np.random.uniform(2,90,n_skus),1),
        "width_in": np.round(np.random.uniform(2,60,n_skus),1),
        "height_in": np.round(np.random.uniform(1,60,n_skus),1),
        "department": np.random.choice(DEPARTMENTS,n_skus)
    })

    # POPULARITY WEIGHT (LONG-TAIL DEMAND)
    catalog["popularity"] = np.random.pareto(1.5, n_skus)
    catalog["popularity"] = catalog["popularity"] / catalog["popularity"].sum()

    catalog.to_excel("product_catalog.xlsx", index=False)
    return catalog

In [32]:
# UTILITY FUNCTIONS

def random_date(year):
    start = datetime(year,1,1)
    end = datetime(year,12,31)
    return start + timedelta(days=random.randint(0,(end-start).days))

def random_zip():
    # GENERATES A RANDOM 5-DIGIT STRING ZIP WITHIN MAINLAND USA (EXCLUDING CA/AK/Islands)
    while True:
        # RANGE COVERS MA (01001) TO WA (99403)
        z_int = np.random.randint(1001, 99404)
        
        # EXCLUDE CA (90000-96199) AND AK (99500-99999)
        if not (90000 <= z_int <= 96199) and not (99500 <= z_int <= 99999):
            return str(z_int).zfill(5)

def random_latlon():
    return (
        np.round(np.random.uniform(25,49),6),
        np.round(np.random.uniform(-124,-67),6)
    )

# TIME UTILITIES
def random_time_daytime():
    # 8 AM – 8 PM WINDOW FOR DELIVERIES AND PICKUPS.
    return datetime(
        2000, 1, 1,
        random.randint(8, 20),
        random.randint(0, 59)
    ).time()

def combine_date_time(date_obj, time_obj):
    dt = datetime.combine(date_obj.date(), time_obj)
    return dt.strftime('%m/%d/%Y %I:%M %p')

def random_time_any():
    return datetime(
        2000, 1, 1,
        random.randint(7, 22),
        random.randint(0, 59)
    ).time()

# ZIP → COORDINATE UTILITY
def zip_to_latlon(zip_code):
    # GENERATES A PLAUSIBLE LAT/LON FOR A GIVEN US ZIP.
    # NOT EXACT, BUT GEOGRAPHICALLY CONSISTENT.

    z = int(zip_code)

    # ROUGH REGIONAL MAPPING BY ZIP PREFIX
    if 1000 <= z < 29999:      # EAST COAST
        lat = np.random.uniform(35, 45)
        lon = np.random.uniform(-85, -70)

    elif 30000 <= z < 59999:   # MIDWEST / SOUTH
        lat = np.random.uniform(30, 45)
        lon = np.random.uniform(-100, -85)

    elif 60000 <= z < 89999:   # CENTRAL / MOUNTAIN
        lat = np.random.uniform(30, 45)
        lon = np.random.uniform(-115, -100)

    else:                      # WEST COAST
        lat = np.random.uniform(32, 49)
        lon = np.random.uniform(-124, -117)

    return round(lat, 6), round(lon, 6)

# DISTANCE + ZONE FUNCTIONS
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8
    phi1,phi2 = math.radians(lat1),math.radians(lat2)
    dphi = math.radians(lat2-lat1)
    dlambda = math.radians(lon2-lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R*2*math.atan2(math.sqrt(a),math.sqrt(1-a))

def pricing_zone(dist):
    if dist<=50:return 2
    if dist<=150:return 3
    if dist<=300:return 4
    if dist<=600:return 5
    if dist<=1000:return 6
    if dist<=1400:return 7
    return 8

In [33]:
# RATE CARDS (ALL SERVICES) - CHATGPT CREATED 3/17 12:20 AM

# HOME DELIVERY (Residential Ground)
HOME = {
    1: {2:8.00,3:8.45,4:9.10,5:9.85,6:10.60,7:11.35,8:12.25},
    2: {2:8.35,3:8.95,4:9.75,5:10.55,6:11.40,7:12.25,8:13.25},
    3: {2:8.75,3:9.50,4:10.35,5:11.20,6:12.10,7:13.15,8:14.25},
    4: {2:9.20,3:10.05,4:11.00,5:11.95,6:13.00,7:14.15,8:15.35},
    5: {2:9.75,3:10.70,4:11.80,5:12.85,6:14.00,7:15.25,8:16.55},
    10:{2:12.40,3:13.85,4:15.65,5:17.25,6:19.40,7:21.75,8:24.10},
    20:{2:16.95,3:19.25,4:22.65,5:25.40,6:29.15,7:33.90,8:38.60},
    50:{2:28.75,3:33.95,4:39.25,5:45.10,6:52.55,7:60.95,8:69.75}
}

# GROUND (Commercial)
GROUND = {
    1:{2:7.75,3:8.10,4:8.70,5:9.35,6:10.15,7:11.00,8:11.90},
    2:{2:8.10,3:8.60,4:9.30,5:10.05,6:10.95,7:11.85,8:12.85},
    3:{2:8.55,3:9.10,4:9.90,5:10.75,6:11.75,7:12.75,8:13.85},
    4:{2:8.95,3:9.65,4:10.55,5:11.50,6:12.60,7:13.75,8:14.95},
    5:{2:9.40,3:10.20,4:11.20,5:12.35,6:13.55,7:14.80,8:16.10},
    10:{2:11.95,3:13.25,4:15.05,5:16.60,6:18.85,7:21.10,8:23.45},
    20:{2:16.35,3:18.45,4:21.75,5:24.30,6:27.85,7:32.25,8:36.90},
    50:{2:27.80,3:32.50,4:37.60,5:43.10,6:50.25,7:58.30,8:66.95}
}

# EXPRESS SAVER (3 BUSINESS DAYS)
EXPRESS = {
    1:{2:11.25,3:11.95,4:12.75,5:13.95,6:15.25,7:16.90,8:18.75},
    2:{2:12.10,3:12.90,4:13.85,5:15.20,6:16.75,7:18.55,8:20.60},
    3:{2:13.10,3:14.00,4:15.10,5:16.60,6:18.30,7:20.25,8:22.45},
    5:{2:15.80,3:17.05,4:18.60,5:20.55,6:22.85,7:25.35,8:28.15},
    10:{2:21.50,3:23.60,4:26.05,5:29.05,6:32.65,7:36.90,8:41.75}
}

# 2DAY (PM Delivery Standard)
TWO_DAY = {
    1:{2:14.75,3:15.60,4:16.90,5:18.40,6:20.25,7:22.40,8:24.90},
    2:{2:15.60,3:16.60,4:18.10,5:19.85,6:21.90,7:24.25,8:27.00},
    3:{2:16.75,3:17.90,4:19.55,5:21.55,6:23.90,7:26.60,8:29.75},
    5:{2:19.95,3:21.45,4:23.45,5:25.95,6:28.95,7:32.35,8:36.25},
    10:{2:26.60,3:29.15,4:32.35,5:36.15,6:40.65,7:45.95,8:51.95}
}

# STANDARD OVERNIGHT
OVERNIGHT = {
    1:{2:21.50,3:23.40,4:25.75,5:28.60,6:31.95,7:35.75,8:39.95},
    2:{2:22.90,3:25.00,4:27.60,5:30.75,6:34.40,7:38.55,8:43.10},
    3:{2:24.55,3:26.85,4:29.70,5:33.15,6:37.05,7:41.55,8:46.65},
    5:{2:28.75,3:31.50,4:35.05,5:39.20,6:44.10,7:49.75,8:56.15},
    10:{2:38.60,3:42.80,4:47.80,5:53.75,6:60.75,7:69.10,8:78.90}
}

In [34]:
# BOX PACKAGING (20 MOST POPULAR)

BOXES = [
    (4,4,4),(6,6,6),(8,6,4),(8,8,8),(10,8,6),
    (10,10,10),(12,9,6),(12,10,6),(12,12,6),(12,12,12),
    (14,14,14),(15,15,6),(16,12,8),(16,16,16),
    (18,12,12),(18,18,16),(18,18,24),
    (20,16,14),(20,20,12),(20,20,20)
]

BOXES = sorted(BOXES, key=lambda b: b[0]*b[1]*b[2])

In [35]:
# DIMENSION LOGIC

def dim_flag(weight, length, width, height, service):

    volume = length * width * height
    longest = max(length,width,height)

    if volume > 700 or longest > 74:
        return "Y"

    if service == "FedEx Home Delivery" and weight > 40:
        return "Y"

    if service == "FedEx Ground Economy" and weight > 149.99:
        return "Y"

    return "N"

In [36]:
# ORDER GENERATION

def generate_year(year, catalog, start_order):

    rows = []
    order_num = start_order

    num_lines_per_order = np.random.randint(1, 8, ORDERS_PER_YEAR)

    fulfillment_choices = np.random.choice(
        ["BOPIS", "STH"],
        ORDERS_PER_YEAR,
        p=[0.60, 0.40]
    )

    store_indices = np.random.randint(0, len(STORES), ORDERS_PER_YEAR)

    # CUSTOMER POOL
    N_CUSTOMERS = 105_155
    customer_ids = np.arange(1, N_CUSTOMERS + 1)

    weights = np.random.pareto(1.3, N_CUSTOMERS)
    weights /= weights.sum()

    order_customers = np.random.choice(customer_ids, ORDERS_PER_YEAR, p=weights)

    for i in range(ORDERS_PER_YEAR):

        n = num_lines_per_order[i]
        fulfillment = fulfillment_choices[i]
        
        store = STORES[store_indices[i]]
        # BOPIS-ONLY STORES CANNOT SHIP-TO-HOME
        if store_indices[i] >= 10:
            fulfillment = "BOPIS"
        
        order_date = random_date(year)

        customer_id = order_customers[i]
        customer_name = f"Customer{customer_id}"
        customer_email = f"customeremailpracticellc{customer_id}@email.com"

        items = catalog.sample(n, weights="popularity")

        # DESTINATION
        if fulfillment == "STH":
            dest_zip = random_zip()
            dest_lat, dest_lon = zip_to_latlon(dest_zip)
            curbside = "N"
            ship_date = order_date + timedelta(days=random.randint(1, 5))
        else:
            dest_zip = np.nan
            dest_lat = np.nan
            dest_lon = np.nan
            curbside = np.random.choice(["Y", "N"], p=[0.4, 0.6])

            ship_date = (
                order_date if random.random() < 0.45
                else order_date + timedelta(days=random.randint(1, 5))
            )

        order_dt = combine_date_time(order_date, random_time_any())
        ship_dt = combine_date_time(ship_date, random_time_daytime())

        # ORDER LINES
        for line_idx, (_, item) in enumerate(items.iterrows(), 1):

            qty = np.random.randint(1, 5)

            subtotal = round(item.price * qty, 2)
            total = round(subtotal * 1.0625, 2)

            rows.append({
                "Order Number": order_num,
                "Order Date": order_dt,

                "SKU": item.sku,
                "SKU Description": item.sku_description,
                "Prime Line Number": line_idx,
                "Order Line Units Sold": qty,

                "Total with tax": total,
                "Total without tax": subtotal,

                "Fulfillment Method": fulfillment,
                "Curbside": curbside,
                "Ship Node Description": store[0],

                "Customer Name": customer_name,
                "Customer Email": customer_email,

                "Destination ZIP": dest_zip,
                "Destination Latitude": dest_lat,
                "Destination Longitude": dest_lon,

                "Shipment Status Ship Date": ship_dt
            })

        order_num += 1

    df = pd.DataFrame(rows)

    # SHIPPED VS CANCELLED
    shipped_orders = df["Order Number"].drop_duplicates().sample(SHIPPED_PER_YEAR)

    df["Order Line Status"] = np.where(
        df["Order Number"].isin(shipped_orders),
        "Shipped",
        "Cancelled"
    )

    cancel_mask = df["Order Line Status"] == "Cancelled"

    cancel_offsets = np.random.randint(1, 4, cancel_mask.sum())

    cancel_dates = (
        pd.to_datetime(df.loc[cancel_mask, "Order Date"], format="%m/%d/%Y %I:%M %p")
        + pd.to_timedelta(cancel_offsets, unit="D")
    )

    cancel_times = [
        random_time_daytime()
        for _ in range(cancel_mask.sum())
    ]

    df.loc[cancel_mask, "Shipment Status Ship Date"] = [
        combine_date_time(d, t)
        for d, t in zip(cancel_dates, cancel_times)
    ]

    # REMOVE DESTINATION FOR CANCELLED ORDERS
    df.loc[cancel_mask, ["Destination ZIP",
                         "Destination Latitude",
                         "Destination Longitude"]] = np.nan

    df.to_excel(f"demand_{year}.xlsx", index=False)

    fulfilled = df[df["Order Line Status"] == "Shipped"].copy()
    fulfilled.to_excel(f"fulfilled_{year}.xlsx", index=False)

    return df, fulfilled, order_num

In [37]:
# PACKING LOGIC 

def build_packages(group):

    items = []

    # EXPAND QUANTITIES INTO INDIVIDUAL ITEMS
    for _, r in group.iterrows():
        for _ in range(r["Order Line Units Sold"]):
            items.append({
                "weight": r["weight_lb"],
                "L": r["length_in"],
                "W": r["width_in"],
                "H": r["height_in"]
            })

    packages = []

    while items:

        pkg_items = []
        used_volume = 0
        max_L = max_W = max_H = 0
        total_weight = 0

        remaining = []

        for it in items:

            new_L = max(max_L, it["L"])
            new_W = max(max_W, it["W"])
            new_H = max(max_H, it["H"])

            fits_box = None
            for box in BOXES:
                if new_L <= box[0] and new_W <= box[1] and new_H <= box[2]:
                    fits_box = box
                    break

            if fits_box:
                pkg_items.append(it)
                max_L, max_W, max_H = new_L, new_W, new_H
                total_weight += it["weight"]
            else:
                remaining.append(it)

        # IF NOTHING FITS TOGETHER → SHIP LARGEST ITEM ALONE
        if not pkg_items:
            it = items[0]
            packages.append({
                "weight": it["weight"],
                "L": it["L"],
                "W": it["W"],
                "H": it["H"]
            })
            items = items[1:]
            continue

        packages.append({
            "weight": total_weight,
            "L": max_L,
            "W": max_W,
            "H": max_H
        })

        items = remaining

    return packages

In [38]:
# WEIGHT BRACKET LOOKUP
def get_weight_bracket(weight, rate_card):
    brackets = sorted(rate_card.keys())
    for b in brackets:
        if weight <= b:
            return b
    return max(brackets)

# RATE LOOKUP
def get_base_rate(service, weight, zone):
    if service == "FedEx Home Delivery":
        table = HOME
    elif service == "FedEx Ground":
        table = GROUND
    elif service == "FedEx Express Saver":
        table = EXPRESS
    elif service == "FedEx 2Day":
        table = TWO_DAY
    elif service == "FedEx Standard Overnight":
        table = OVERNIGHT
    else:
        raise ValueError("Unknown FedEx service")

    bracket = get_weight_bracket(weight, table)
    return table[bracket][zone]

In [39]:
# SHIPPING & FEDEX LOGIC

def create_shipping(year, fulfilled, catalog):

    sth = fulfilled[
        fulfilled["Fulfillment Method"] == "STH"
    ].copy()

    sth = sth.merge(catalog, left_on="SKU", right_on="sku", how="left")

    shipments = []
    
    order_tracking_map = []
    
    for order, g in sth.groupby("Order Number"):
    
        packages = build_packages(g)
    
        store = next(s for s in STORES if s[0] == g.iloc[0]["Ship Node Description"])
    
        dest_zip = g.iloc[0]["Destination ZIP"]
        dest_lat = g.iloc[0]["Destination Latitude"]
        dest_lon = g.iloc[0]["Destination Longitude"]
    
        dist = haversine(store[4], store[5], dest_lat, dest_lon)
        z = pricing_zone(dist)
    
        ship_dt = g.iloc[0]["Shipment Status Ship Date"]

        order_trackings = [] 
        
        for p_idx, pkg in enumerate(packages):
    
            weight = pkg["weight"]
            L = math.ceil(pkg["L"])
            W = math.ceil(pkg["W"])
            H = math.ceil(pkg["H"])
    
            cubic = L * W * H
            dim_w = math.ceil(cubic / 139)
            billable = max(weight, dim_w)
    
            tracking = f"TRK{year}{order:08}{p_idx:02}"
            order_trackings.append(tracking)
    
            # RESIDENTIAL PROBABILITY
            residential = random.random() < 0.75
    
            if residential:
                service = random.choice([
                    "FedEx Home Delivery",
                    "FedEx Express Saver",
                    "FedEx 2Day"
                ])
            else:
                service = random.choice([
                    "FedEx Ground",
                    "FedEx Express Saver",
                    "FedEx 2Day"
                ])
    
            base = get_base_rate(service, billable, z)
            fuel = base * random.uniform(0.12, 0.22)
    
            handling = 0
            if cubic > 17280:
                handling = 105
            elif cubic > 10368:
                handling = 20
    
            charge = round(base + fuel + handling, 2)
    
            delivery_date = datetime.strptime(
                ship_dt, "%m/%d/%Y %I:%M %p"
            ) + timedelta(days=random.randint(1, 5))
    
            delivery_dt = combine_date_time(delivery_date, random_time_daytime())
    
            shipments.append({
    
                "Shipment Tracking Number": tracking,
                "Order Number": order,
    
                "Origin Store": store[0],
                "Origin ZIP": store[3],
                "Origin Latitude": store[4],
                "Origin Longitude": store[5],
    
                "Recipient Postal Code": dest_zip,
                "Destination Latitude": dest_lat,
                "Destination Longitude": dest_lon,
    
                "Distance Miles": round(dist, 1),
                "Pricing Zone": z,
    
                "Service Description": service,
    
                "Ship Date": ship_dt,
                "Delivery Date": delivery_dt,
    
                "Original Weight (Pounds)": round(weight, 2),
                "Shipment Rated Weight (Pounds)": billable,
    
                "Dimmed Height (in)": H,
                "Dimmed Width (in)": W,
                "Dimmed Length (in)": L,
    
                "Shipment DIM Flag": "Y" if billable > weight else "N",
    
                "Net Charge Amount USD": charge
            })
    
        # BUILD ORDER ↔ TRACKING RELATIONSHIP
        for t in order_trackings:
            order_tracking_map.append({
                "Order Number": order,
                "Tracking Number": t
            })    

    # PACKAGE LEVEL
    fedex = pd.DataFrame(shipments)
    fedex.to_excel(f"fedex_parcel_{year}.xlsx", index=False)

    # COMPANY SHIPPING FILE (NOW ALSO PACKAGE LEVEL)
    company = pd.DataFrame(order_tracking_map)
    company.to_excel(
        f"company_shipping_{year}.xlsx",
        index=False
    )


In [40]:
# STORES FILE

pd.DataFrame(
    STORES,
    columns=["Store","City","State","ZIP","Latitude","Longitude"]
).to_excel("stores.xlsx", index=False)

In [41]:
# MAIN

catalog = create_catalog()

d2024, f2024, next_order = generate_year(2024, catalog, START_ORDER_NUM)
d2025, f2025, _ = generate_year(2025, catalog, next_order)

create_shipping(2024, f2024, catalog)
create_shipping(2025, f2025, catalog)

In [42]:
print("FULL ENTERPRISE DATASET GENERATED")

FULL ENTERPRISE DATASET GENERATED


In [43]:
# MAPPING FEDEX FILE FOR FEDEX SAMPLE
fedex2024 = pd.read_excel("fedex_parcel_2024.xlsx")
fedex2025 = pd.read_excel("fedex_parcel_2025.xlsx")


samples = {
    "demand_2024_SAMPLE.xlsx": d2024,
    "fulfilled_2024_SAMPLE.xlsx": f2024,
    "fedex_parcel_2024_SAMPLE.xlsx": fedex2024,
    "demand_2025_SAMPLE.xlsx": d2025,
    "fulfilled_2025_SAMPLE.xlsx": f2025,
    "fedex_parcel_2025_SAMPLE.xlsx": fedex2025,
}

for name, df in samples.items():
    df.head(5000).to_excel(name, index=False)
    print("Created:", name)

print("\nAll samples ready")

Created: demand_2024_SAMPLE.xlsx
Created: fulfilled_2024_SAMPLE.xlsx
Created: fedex_parcel_2024_SAMPLE.xlsx
Created: demand_2025_SAMPLE.xlsx
Created: fulfilled_2025_SAMPLE.xlsx
Created: fedex_parcel_2025_SAMPLE.xlsx

All samples ready


In [44]:
# 10 OUTPUTS EXPECTED

# 2 REFERENCE: PRODUCT CATALOG, STORES
# 4 ORDERS: DEMAND 2024, DEMAND 2025, FULFILLED 2024, FULFILLED 2025
# 2 SHIPPING: COMPANY SHIPPING 2024, COMPANY SHIPPING 2025
# 2 PARCEL: FEDEX PARCEL 2024, FEDEX PARCEL 2025